In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df= pd.read_csv('train.csv')

In [6]:
df.shape

(404290, 6)

In [8]:
df.head()


,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [10]:
new_df = df.sample(40000)

In [12]:
new_df.isnull().sum()

id              0
qid1            0
qid2            0
question1       0
question2       0
is_duplicate    0
dtype: int64

In [14]:
new_df.duplicated().sum()

np.int64(0)

In [16]:
new_df.head()

,id,qid1,qid2,question1,question2,is_duplicate
332521,332521,459516,66045,How are download managers able to download fil...,How do I download a torrent with IDM more than...,0
200753,200753,302601,301254,What traffic laws in Antigua and Barbuda are p...,What if jio is not registered within 48 hours?,0
202914,202914,305361,305362,What are some characteristics of the Mesolithi...,Why do I wish I was born in a different time p...,0
362289,362289,492192,115281,What are tips and hacks for new ride-sharing d...,What are tips and hacks for new ride-sharing d...,0
28912,28912,53554,53555,How would one begin to create their own operat...,How can I create an operating system?,1


In [18]:
ques_df = new_df[['question1', 'question2']]
ques_df.head()

,question1,question2
332521,How are download managers able to download fil...,How do I download a torrent with IDM more than...
200753,What traffic laws in Antigua and Barbuda are p...,What if jio is not registered within 48 hours?
202914,What are some characteristics of the Mesolithi...,Why do I wish I was born in a different time p...
362289,What are tips and hacks for new ride-sharing d...,What are tips and hacks for new ride-sharing d...
28912,How would one begin to create their own operat...,How can I create an operating system?


In [20]:
from sklearn.feature_extraction.text import CountVectorizer
questions = list(ques_df['question1']) + list(ques_df['question2'])
cv = CountVectorizer(max_features = 3000)
q1_arr, q2_arr = np.vsplit(cv.fit_transform(questions).toarray(), 2)

In [22]:
temp_df1 = pd.DataFrame(q1_arr, index= ques_df.index)
temp_df2 = pd.DataFrame(q2_arr, index= ques_df.index)
temp_df = pd.concat([temp_df1, temp_df2], axis=1)
temp_df.shape

(40000, 6000)

In [24]:
temp_df['is_duplicate'] = new_df['is_duplicate']

In [26]:
temp_df.head()

,0,1,2,3,4,5,6,7,8,9,...,2991,2992,2993,2994,2995,2996,2997,2998,2999,is_duplicate
332521,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
200753,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
202914,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
362289,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
28912,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [28]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(temp_df.iloc[:,0:-1].values,temp_df.iloc[:,-1].values,test_size=0.2,random_state=1)


In [29]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test,y_pred)

0.74875

In [30]:
from xgboost import XGBClassifier
xgb = XGBClassifier()
xgb.fit(X_train,y_train)
y_pred = xgb.predict(X_test)
accuracy_score(y_test,y_pred)

0.73125